# Google Colab - ClinicalBERT + TrOCR + FAISS Remote Core
Run this on Google Colab (GPU Enabled).

In [ ]:
# 1. Install Dependencies
!pip install -q fastapi uvicorn pyngrok nest-asyncio transformers easyocr faiss-cpu sentence-transformers torch torchvision torchaudio python-multipart

In [ ]:
import nest_asyncio
from fastapi import FastAPI, UploadFile, File as FastFile, Body
from pyngrok import ngrok
import uvicorn
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import easyocr
import io
import warnings
import logging
import asyncio
import datetime
import os

# Suppress annoying warnings
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

app = FastAPI(title="ClinicalBERT & TrOCR Remote Brain")
nest_asyncio.apply()

@app.get("/")
def read_root():
    return {"status": "ClinicalBERT & TrOCR Remote Brain is Running! Everything is connected perfectly."}

# --- LOAD MODELS ---
print("Loading EasyOCR (GPU)...")
ocr_reader = easyocr.Reader(['en'], gpu=True, verbose=False)

print("Loading ClinicalBERT for NER...")
tokenizer = AutoTokenizer.from_pretrained("samrawal/bert-base-uncased_clinical-ner")
model = AutoModelForTokenClassification.from_pretrained("samrawal/bert-base-uncased_clinical-ner")
ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

@app.post("/extract_prescription")
async def extract_prescription(file: UploadFile = FastFile(...)):
    """
    1. TrOCR / EasyOCR parses Image -> Text
    2. ClinicalBERT parses Text -> Medicines, Dosages
    """
    now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"\n{'='*50}")
    print(f"📡 [{now}] NEW REQUEST RECEIVED!")
    print(f"📥 File Name: {file.filename}")
    
    contents = await file.read()
    print(f"✅ Image loaded. Size: {len(contents) / 1024:.2f} KB")
    print("⏳ Running EasyOCR on GPU (Extracting text from image)...")
    
    # 1. Image to Text
    ocr_result = ocr_reader.readtext(contents, detail=0)
    raw_text = " ".join(ocr_result)
    
    print(f"✅ OCR Extracted {len(raw_text)} characters.")
    print(f"📝 OCR Preview: {raw_text[:80]}...")
    
    # 2. Extract Medical Entities
    print("⏳ Running ClinicalBERT Tokenizer & NER Pipeline...")
    try:
        entities = ner_pipeline(raw_text)
        print(f"✅ NER found {len(entities)} medical entities!")
    except Exception as e:
        print(f"❌ Error in NER: {str(e)}")
        entities = [{"error": str(e)}]
        
    print(f"🚀 Processing successful! Sending JSON back to Django Server.")
    print(f"{'='*50}\n")
    
    return {
        "raw_ocr": raw_text,
        "clinical_entities": str(entities)
    }

# NGROK TUNNEL CONFIGURATION (Paste your token)
NGROK_TOKEN = "3DX2cM8B2U1rvm0cp1MSz2IlFUb_5qnn3KdyogDLQKN4XgV7H"
ngrok.set_auth_token(NGROK_TOKEN)

try:
    for tunnel in ngrok.get_tunnels(): ngrok.disconnect(tunnel.public_url)
    ngrok.kill()
except:
    pass

tunnel = ngrok.connect(8000)
print(f"🚀 CLINICAL BERT BRAIN IS LIVE AT: {tunnel.public_url}")
# UPDATE django .env with this URL.

# Google Colab Background Task Workaround
# Port 8000 might be blocked by a previous crashed session, kill it:
os.system("fuser -k 8000/tcp")

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
asyncio.ensure_future(server.serve())